# ETL Process: Data Warehouse Implementation

This notebook implements the complete ETL process to transform cleaned source data into a star schema data warehouse.

## Process Overview
1. **Extract**: Load cleaned source data
2. **Transform**: Create dimensional and fact tables
3. **Validate**: Quality checks and referential integrity
4. **Load**: Export to CSV and generate reports

## Setup: Imports and Configuration

In [93]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_execution.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
SOURCE_DIR = "."
OUTPUT_DIR = "./dw_output"
DIMS_DIR = os.path.join(OUTPUT_DIR, "dimensions")
FACTS_DIR = os.path.join(OUTPUT_DIR, "facts")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

# Create output directories
for dir_path in [DIMS_DIR, FACTS_DIR, LOGS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

logger.info("ETL Process Started")
logger.info(f"Output directories created: {OUTPUT_DIR}")

2026-04-03 23:49:26,645 - INFO - ETL Process Started
2026-04-03 23:49:26,646 - INFO - Output directories created: ./dw_output


## Phase 1: Extraction

Load all cleaned source datasets

In [94]:
# Load raw source data
logger.info("=" * 60)
logger.info("PHASE 1: EXTRACTION & CLEANING")
logger.info("=" * 60)

try:
    # Load raw files
    df_pop_raw = pd.read_csv("ine_population_data.csv", encoding="latin1", sep=";", skiprows=3)
    logger.info(f"✓ Population data extracted: {len(df_pop_raw)} rows")
    
    df_substations_raw = pd.read_csv("postos-transformacao-distribuicao.csv", sep=";")
    logger.info(f"✓ Substations data extracted: {len(df_substations_raw)} rows")
    
    df_weather_sp_raw = pd.read_csv("porto_serra_do_pilar_weather.csv")
    logger.info(f"✓ Weather (Serra do Pilar) extracted: {len(df_weather_sp_raw)} rows")
    
    df_weather_pr_raw = pd.read_csv("porto_pedras_rubras_weather.csv")
    logger.info(f"✓ Weather (Pedras Rubras) extracted: {len(df_weather_pr_raw)} rows")
    
    df_consumption_pt1_raw = pd.read_csv("consumo_horario_mobilidade_eletrica_01_a_10.csv", sep=";")
    df_consumption_pt2_raw = pd.read_csv("consumo_horario_mobilidade_eletrica_11_a_18.csv", sep=";")
    logger.info(f"✓ Consumption data extracted: {len(df_consumption_pt1_raw) + len(df_consumption_pt2_raw)} rows")
    
    # Raw charging data will be loaded with proper encoding
    df_charging_raw = pd.read_csv(
        "ine_charging_points_number_location.csv", 
        encoding='iso-8859-1',
        sep=';',
        on_bad_lines='skip',
        engine='python'
    )
    logger.info(f"✓ Raw Charging data loaded: {len(df_charging_raw)} rows")
    
    logger.info("Extraction completed successfully")
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
    raise
except Exception as e:
    logger.error(f"Error during extraction: {e}")
    raise

2026-04-03 23:49:26,687 - INFO - ============================================================
2026-04-03 23:49:26,690 - INFO - PHASE 1: EXTRACTION & CLEANING
2026-04-03 23:49:26,691 - INFO - ============================================================
2026-04-03 23:49:26,711 - INFO - ✓ Population data extracted: 180 rows
2026-04-03 23:49:27,038 - INFO - ✓ Substations data extracted: 72349 rows
2026-04-03 23:49:27,043 - INFO - ✓ Weather (Serra do Pilar) extracted: 1520 rows
2026-04-03 23:49:27,046 - INFO - ✓ Weather (Pedras Rubras) extracted: 1520 rows
2026-04-03 23:49:39,733 - INFO - ✓ Consumption data extracted: 9315214 rows
2026-04-03 23:49:39,742 - INFO - ✓ Raw Charging data loaded: 7 rows
2026-04-03 23:49:39,743 - INFO - Extraction completed successfully


## 1.1: Clean Population Data

In [95]:
# Clean population data
df_pop = df_pop_raw.copy()
df_pop.drop(0, inplace=True)

# Rename columns
df_pop.columns = [
    "year", "region", "growth_rate", "density", 
    "cities", "freguesias", "vilas", "extra"
]

# Fill year forward (for merged cells in original data)
df_pop["year"] = df_pop["year"].ffill()

# Select data of interest (first 73 rows contain Porto Metropolitan Area data)
df_pop = df_pop.iloc[0:72]

# Split region into code and name
df_pop[["region_code", "region_name"]] = df_pop["region"].str.split(":", n=1, expand=True)
df_pop = df_pop.drop('region', axis=1)

# Remove extra column and clean region names
df_pop = df_pop.drop('extra', axis=1)
df_pop["region_name"] = df_pop["region_name"].str.strip()

# Get unique regions for filtering other datasets
unique_regions = df_pop["region_name"].dropna().unique().tolist()

logger.info(f"✓ Population data cleaned: {len(df_pop)} rows")
logger.info(f"  Columns: {df_pop.columns.tolist()}")
logger.info(f"  Regions: {len(unique_regions)} municipalities")

2026-04-03 23:49:39,773 - INFO - ✓ Population data cleaned: 72 rows
2026-04-03 23:49:39,774 - INFO -   Columns: ['year', 'growth_rate', 'density', 'cities', 'freguesias', 'vilas', 'region_code', 'region_name']
2026-04-03 23:49:39,775 - INFO -   Regions: 18 municipalities


## 1.2: Clean Substations Data

In [96]:
# Clean substations data - filter for Porto Metropolitan Area
df_substations = df_substations_raw[
    df_substations_raw["Municipality"].isin(unique_regions)
]

logger.info(f"✓ Substations data cleaned: {len(df_substations)} rows")
logger.info(f"  Filtered to Porto Metropolitan Area")

2026-04-03 23:49:39,812 - INFO - ✓ Substations data cleaned: 8412 rows
2026-04-03 23:49:39,813 - INFO -   Filtered to Porto Metropolitan Area


## 1.3: Clean Weather Data

In [97]:
# Clean weather data for both stations
for df_raw, station_name in [(df_weather_sp_raw, "Serra do Pilar"), 
                              (df_weather_pr_raw, "Pedras Rubras")]:
    # Convert date column
    df_raw["date"] = pd.to_datetime(df_raw["date"])
    
    # Extract datetime components
    df_raw["year"] = df_raw["date"].dt.year
    df_raw["month"] = df_raw["date"].dt.month
    df_raw["day"] = df_raw["date"].dt.day
    df_raw["time"] = df_raw["date"].dt.time
    
    # Drop columns with all null values
    df_raw.drop(columns=["snow", "wdir", "tsun"], inplace=True)
    
    logger.info(f"  Weather ({station_name}) cleaned: {len(df_raw)} rows")

# Assign cleaned data
df_weather_sp = df_weather_sp_raw
df_weather_pr = df_weather_pr_raw

logger.info(f"✓ Weather data cleaned for both stations")

2026-04-03 23:49:39,840 - INFO -   Weather (Serra do Pilar) cleaned: 1520 rows
2026-04-03 23:49:39,852 - INFO -   Weather (Pedras Rubras) cleaned: 1520 rows
2026-04-03 23:49:39,855 - INFO - ✓ Weather data cleaned for both stations


## 1.4: Clean Consumption Data

In [98]:
# Clean consumption data - combine and filter for Porto Metropolitan Area
df_consumption_pt1 = df_consumption_pt1_raw[
    df_consumption_pt1_raw["Municipality"].isin(unique_regions)
]
df_consumption_pt2 = df_consumption_pt2_raw[
    df_consumption_pt2_raw["Municipality"].isin(unique_regions)
]

# Combine both parts
df_consumption = pd.concat(
    [df_consumption_pt1, df_consumption_pt2], 
    axis=0, 
    ignore_index=True
)

logger.info(f"✓ Consumption data cleaned: {len(df_consumption)} rows")
logger.info(f"  Combined from 2 datasets and filtered to Porto Metropolitan Area")

2026-04-03 23:49:40,456 - INFO - ✓ Consumption data cleaned: 1027232 rows
2026-04-03 23:49:40,457 - INFO -   Combined from 2 datasets and filtered to Porto Metropolitan Area


## 1.5: Clean Charging Data

In [99]:
def clean_charging_data(filename):
    """
    Clean the raw INE charging points data.
    
    File structure:
    - Line 8: Months (Dezembro de 2024 in col 1, empty cols 2-4, Novembro de 2024 in col 6, etc.)
    - Line 10: Socket types (Total, Mennekes, CHAdeMO, CCS, Industrial repeating)
    - Line 12+: Data rows (location in col 0, values in cols 1+)
    - Each month takes 5 columns for the 5 socket types
    """
    
    with open(filename, 'r', encoding='iso-8859-1') as f:
        lines = f.readlines()
    
    # Extract header information
    months_row = [x.strip() for x in lines[8].split(';')]
    socket_types_row = [x.strip() for x in lines[10].split(';')]
    
    # Build month-year mapping for each column position
    col_to_month = {}
    month_map = {
        'Janeiro': 1, 'Fevereiro': 2, 'Março': 3, 'Abril': 4,
        'Maio': 5, 'Junho': 6, 'Julho': 7, 'Agosto': 8,
        'Setembro': 9, 'Outubro': 10, 'Novembro': 11, 'Dezembro': 12
    }
    
    current_month_year = None
    for col_idx, val in enumerate(months_row):
        if val and 'de 20' in val:
            parts = val.split()
            month_name = parts[0]
            year = int(parts[-1])
            month_num = month_map.get(month_name, 1)
            current_month_year = (year, month_num)
        
        if current_month_year:
            col_to_month[col_idx] = current_month_year
    
    # Process data rows
    data_rows = []
    for line_idx in range(12, len(lines)):
        line = lines[line_idx].strip()
        if not line:
            continue
        
        cols = [x.strip() for x in line.split(';')]
        location = cols[0]
        
        if not location:
            continue
        
        # Remove location code
        if ':' in location:
            location = location.split(':', 1)[1].strip()
        
        # Process each value column
        for col_idx, val in enumerate(cols[1:], start=1):
            if col_idx not in col_to_month or col_idx >= len(socket_types_row):
                continue
            
            socket_type = socket_types_row[col_idx]
            year, month = col_to_month[col_idx]
            
            # Parse value
            if '- -' in val or val == '' or val == 'nan':
                count = None
            else:
                try:
                    count = int(val)
                except (ValueError, TypeError):
                    count = None
            
            data_rows.append({
                'location': location,
                'year': year,
                'month': month,
                'outlet_type': socket_type,
                'charging_points': count
            })
    
    df_cleaned = pd.DataFrame(data_rows)
    
    if len(df_cleaned) > 0:
        df_cleaned = df_cleaned.drop_duplicates()
        df_cleaned = df_cleaned.sort_values(['year', 'month', 'location'], ignore_index=True)
    
    return df_cleaned

# Clean the raw charging data
df_charging = clean_charging_data("ine_charging_points_number_location.csv")

logger.info(f"✓ Charging data cleaned: {len(df_charging)} rows")
if len(df_charging) > 0:
    logger.info(f"  Date range: {df_charging['year'].min()}-{df_charging['month'].min():02d} to {df_charging['year'].max()}-{df_charging['month'].max():02d}")
    logger.info(f"  Locations: {df_charging['location'].nunique()}")
    logger.info(f"  Outlet types: {len(df_charging['outlet_type'].unique())}")

2026-04-03 23:49:40,489 - INFO - ✓ Charging data cleaned: 4355 rows
2026-04-03 23:49:40,490 - INFO -   Date range: 2021-01 to 2024-12
2026-04-03 23:49:40,492 - INFO -   Locations: 35
2026-04-03 23:49:40,493 - INFO -   Outlet types: 6


## Phase 2: Transformation

### 2.1 Create DIM_LOCATION

In [100]:
logger.info("=" * 60)
logger.info("PHASE 2: TRANSFORMATION")
logger.info("=" * 60)

# Extract unique municipalities from population data
dim_location = df_pop[['region_code', 'region_name']].drop_duplicates().reset_index(drop=True)

# Rename columns to match dimension schema
dim_location.columns = ['municipality_code', 'municipality_name']

# Assign location_id (surrogate key)
dim_location['location_id'] = range(1, len(dim_location) + 1)

# Add district information (derived from municipality name or mapping)
# For simplicity, we'll use municipality as both municipality and district
# In real scenario, you'd use a mapping table
dim_location['district_code'] = dim_location['municipality_code']
dim_location['district_name'] = dim_location['municipality_name']
dim_location['parish_code'] = None  # Optional data not available in cleaned sources
dim_location['parish_name'] = None

# Reorder columns according to schema
dim_location = dim_location[[
    'location_id',
    'municipality_code',
    'municipality_name',
    'parish_code',
    'parish_name',
    'district_code',
    'district_name'
]]

logger.info(f"✓ DIM_LOCATION created: {len(dim_location)} rows")
display(dim_location.head(10))

2026-04-03 23:49:40,506 - INFO - ============================================================
2026-04-03 23:49:40,507 - INFO - PHASE 2: TRANSFORMATION
2026-04-03 23:49:40,509 - INFO - ============================================================
2026-04-03 23:49:40,523 - INFO - ✓ DIM_LOCATION created: 18 rows


,location_id,municipality_code,municipality_name,parish_code,parish_name,district_code,district_name
0,1,11A,Área Metropolitana do Porto,None,None,11A,Área Metropolitana do Porto
1,2,11A0104,Arouca,None,None,11A0104,Arouca
2,3,11A0107,Espinho,None,None,11A0107,Espinho
3,4,11A1304,Gondomar,None,None,11A1304,Gondomar
4,5,11A1306,Maia,None,None,11A1306,Maia
5,6,11A1308,Matosinhos,None,None,11A1308,Matosinhos
6,7,11A0113,Oliveira de Azeméis,None,None,11A0113,Oliveira de Azeméis
7,8,11A1310,Paredes,None,None,11A1310,Paredes
8,9,11A1312,Porto,None,None,11A1312,Porto
9,10,11A1313,Póvoa de Varzim,None,None,11A1313,Póvoa de Varzim


### 2.2 Create DIM_TIME

In [101]:
# Extract min and max dates from all datasets
dates = []

# From population: year
pop_dates = pd.to_datetime(df_pop['year'].astype(str))
dates.extend(pop_dates.tolist())

# From weather
if 'date' in df_weather_sp.columns:
    dates.extend(pd.to_datetime(df_weather_sp['date']).tolist())
if 'date' in df_weather_pr.columns:
    dates.extend(pd.to_datetime(df_weather_pr['date']).tolist())

# From consumption
if 'Period' in df_consumption.columns:
    dates.extend(pd.to_datetime(df_consumption['Period'], errors='coerce').dropna().tolist())

# From charging
charging_dates = pd.to_datetime(
    df_charging['year'].astype(str) + '-' + 
    df_charging['month'].astype(str) + '-01',
    errors='coerce'
)
dates.extend(charging_dates.tolist())

# Remove None and duplicates
dates = pd.to_datetime([d for d in dates if d is not None]).unique()
dates = sorted(dates)

min_date = pd.Timestamp(dates[0])
max_date = pd.Timestamp(dates[-1])

logger.info(f"Date range: {min_date.date()} to {max_date.date()}")

# Generate complete datetime range
date_range = pd.date_range(start=min_date, end=max_date, freq='H')

dim_time = pd.DataFrame({
    'datetime': date_range
})

# Extract datetime components
dim_time['date'] = dim_time['datetime'].dt.date
dim_time['year'] = dim_time['datetime'].dt.year
dim_time['month'] = dim_time['datetime'].dt.month
dim_time['day'] = dim_time['datetime'].dt.day
dim_time['hour'] = dim_time['datetime'].dt.hour

# Create level keys (surrogate keys for each level)
# Year level key
dim_time['time_id'] = dim_time['year'] - dim_time['year'].min() + 1

# Month level key
dim_time['month_id'] = (
    (dim_time['year'] - dim_time['year'].min()) * 12 + dim_time['month']
)

# Day level key
dim_time['day_id'] = (
    dim_time['datetime'].astype('int64') // (10**9 * 86400)
)

# Hour level key
dim_time['hour_id'] = range(len(dim_time))

# Reorder columns
dim_time = dim_time[[
    'hour_id',
    'day_id',
    'month_id',
    'time_id',
    'hour',
    'day',
    'month',
    'year',
    'date',
    'datetime'
]]

logger.info(f"✓ DIM_TIME created: {len(dim_time)} rows")
logger.info(f"  Time range: {dim_time['datetime'].min()} to {dim_time['datetime'].max()}")
display(dim_time.head(10))

2026-04-03 23:49:40,637 - INFO - Date range: 2021-01-01 to 2026-02-28
C:\Users\iscap\AppData\Local\Temp\ipykernel_44816\3726882397.py:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(start=min_date, end=max_date, freq='H')
2026-04-03 23:49:40,663 - INFO - ✓ DIM_TIME created: 45217 rows
2026-04-03 23:49:40,665 - INFO -   Time range: 2021-01-01 00:00:00 to 2026-02-28 00:00:00


,hour_id,day_id,month_id,time_id,hour,day,month,year,date,datetime
0,0,18628,1,1,0,1,1,2021,2021-01-01,2021-01-01 00:00:00
1,1,18628,1,1,1,1,1,2021,2021-01-01,2021-01-01 01:00:00
2,2,18628,1,1,2,1,1,2021,2021-01-01,2021-01-01 02:00:00
3,3,18628,1,1,3,1,1,2021,2021-01-01,2021-01-01 03:00:00
4,4,18628,1,1,4,1,1,2021,2021-01-01,2021-01-01 04:00:00
5,5,18628,1,1,5,1,1,2021,2021-01-01,2021-01-01 05:00:00
6,6,18628,1,1,6,1,1,2021,2021-01-01,2021-01-01 06:00:00
7,7,18628,1,1,7,1,1,2021,2021-01-01,2021-01-01 07:00:00
8,8,18628,1,1,8,1,1,2021,2021-01-01,2021-01-01 08:00:00
9,9,18628,1,1,9,1,1,2021,2021-01-01,2021-01-01 09:00:00


### 2.3 Create DIM_WEATHER

In [102]:
# Combine weather data from both stations
df_weather_sp['date'] = pd.to_datetime(df_weather_sp['date'])
df_weather_pr['date'] = pd.to_datetime(df_weather_pr['date'])

# Merge both weather sources (average their values)
weather_merged = pd.merge(
    df_weather_sp[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    df_weather_pr[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    on='date',
    how='outer',
    suffixes=('_sp', '_pr')
)

# Calculate averages
dim_weather = pd.DataFrame({
    'date': weather_merged['date'],
    'avg_temperature': weather_merged[['tavg_sp', 'tavg_pr']].mean(axis=1),
    'min_temperature': weather_merged[['tmin_sp', 'tmin_pr']].min(axis=1),
    'max_temperature': weather_merged[['tmax_sp', 'tmax_pr']].max(axis=1),
    'precipitation': weather_merged[['prcp_sp', 'prcp_pr']].sum(axis=1),
    'wind_speed': weather_merged[['wspd_sp', 'wspd_pr']].mean(axis=1),
    'pressure': weather_merged[['pres_sp', 'pres_pr']].mean(axis=1)
})

# Link to time dimension (day level)
# Convert dim_time date to datetime64[ns] to match dim_weather for merge
dim_time_for_merge = dim_time[['date', 'day_id', 'month_id', 'time_id']].drop_duplicates().copy()
dim_time_for_merge['date'] = pd.to_datetime(dim_time_for_merge['date'])

dim_weather = dim_weather.merge(
    dim_time_for_merge,
    on='date',
    how='left'
)

# Assign weather_id
dim_weather['weather_id'] = range(1, len(dim_weather) + 1)
dim_weather.rename(columns={'time_id': 'time_id_ref'}, inplace=True)
dim_weather['time_id'] = dim_weather['day_id']  # Reference day level key

# Reorder columns
dim_weather = dim_weather[[
    'weather_id',
    'time_id',
    'avg_temperature',
    'min_temperature',
    'max_temperature',
    'precipitation',
    'wind_speed',
    'pressure'
]]

logger.info(f"✓ DIM_WEATHER created: {len(dim_weather)} rows")
display(dim_weather.head(10))

2026-04-03 23:49:40,709 - INFO - ✓ DIM_WEATHER created: 1520 rows


,weather_id,time_id,avg_temperature,min_temperature,max_temperature,precipitation,wind_speed,pressure
0,1,18993,17.20,11.6,23.7,0.0,16.90,1024.55
1,2,18994,13.85,10.6,23.7,4.3,8.45,1029.25
2,3,18995,11.95,8.8,17.2,0.3,14.35,1024.90
3,4,18996,12.00,8.5,15.3,9.7,18.35,1017.75
4,5,18997,9.75,7.1,13.4,10.4,10.35,1021.70
5,6,18998,8.20,4.1,13.3,0.3,6.40,1024.35
6,7,18999,10.05,6.4,14.1,0.3,13.05,1031.45
7,8,19000,7.25,2.3,14.1,0.3,6.45,1032.85
8,9,19001,12.35,6.8,13.7,17.5,8.15,1028.25
9,10,19002,12.30,8.9,13.6,2.3,10.25,1026.70


### 2.4 Create DIM_SUBSTATIONS

In [103]:
# Process substations data
dim_substations = df_substations.copy()

# Assign substation_id (surrogate key)
dim_substations['substation_id'] = range(1, len(dim_substations) + 1)

# Rename and map columns to schema
dim_substations = dim_substations.rename(columns={
    'Installation Code': 'installation_code',
    'Municipality': 'municipality_name'
})

# Map to location dimension to get municipality_code
dim_substations = dim_substations.merge(
    dim_location[['municipality_code', 'municipality_name', 'district_name']],
    on='municipality_name',
    how='left'
)

# Assign level keys (using month_id as proxy for data level)
dim_substations['municipality_code_lk'] = dim_substations['municipality_code']

# Select and reorder columns
columns_to_keep = [
    'substation_id',
    'installation_code',
    'municipality_code_lk',
    'municipality_name',
    'district_name'
]

# Add optional columns if they exist
optional_cols = [
    'Geographic Coordinates',  # Geographic coordinates
    'Installed Power [kVA]',
    'Contracted Power [kVA]',
    'Usage Level [%]',
    'Number of Clients',
    'Construction Type'
]

for col in optional_cols:
    if col in dim_substations.columns:
        columns_to_keep.append(col)

dim_substations = dim_substations[columns_to_keep]

# Rename to match schema
rename_map = {
    'municipality_code_lk': 'municipality_code',
    'Geographic Coordinates': 'geographic_coordinates',
    'Installed Power [kVA]': 'installed_power',
    'Contracted Power [kVA]': 'contracted_power',
    'Usage Level [%]': 'usage_level',
    'Number of Clients': 'number_of_clients',
    'Construction Type': 'construction_type'
}

dim_substations = dim_substations.rename(columns=rename_map)

logger.info(f"✓ DIM_SUBSTATIONS created: {len(dim_substations)} rows")
display(dim_substations.head(10))

2026-04-03 23:49:40,753 - INFO - ✓ DIM_SUBSTATIONS created: 8412 rows


,substation_id,installation_code,municipality_code,municipality_name,district_name,geographic_coordinates,installed_power,contracted_power,usage_level,number_of_clients,construction_type
0,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
1,2,1316D2004400,11A1316,Vila do Conde,Vila do Conde,"41.336527228320506, -8.645671024999814",400,1610.01,20%-39%,216,Cabine alta
2,3,1316D2016800,11A1316,Vila do Conde,Vila do Conde,"41.27164654205575, -8.724611320666718",630,2047.95,20%-39%,336,Cabine baixa em edifício próprio
3,4,1316D2057000,11A1316,Vila do Conde,Vila do Conde,"41.279717670991126, -8.679046573660683",400,N/D,0%-19%,<20,Cabine pré-fabricada
4,5,1317D2070300,11A1317,Vila Nova de Gaia,Vila Nova de Gaia,"41.04344892828184, -8.542996018121055",250,640.00,20%-39%,71,Cabine baixa em edifício próprio
5,6,1317D2051700,11A1317,Vila Nova de Gaia,Vila Nova de Gaia,"41.11455052390216, -8.61403627288223",630,1178.46,0%-19%,188,Cabine baixa em edifício próprio
6,7,1317D2033500,11A1317,Vila Nova de Gaia,Vila Nova de Gaia,"41.13972799931655, -8.636187844159922",630,592.77,0%-19%,91,Cabine baixa em edifício próprio
7,8,1308D2023400,11A1308,Matosinhos,Matosinhos,"41.25329240728268, -8.721106859919177",400,2580.12,60%-79%,373,Cabine alta
8,9,1308D2071200,11A1308,Matosinhos,Matosinhos,"41.20898585173012, -8.687992786282576",1890,4095.66,20%-39%,185,Cabine baixa integrada em edifício
9,10,1308D2009700,11A1308,Matosinhos,Matosinhos,"41.196189239991355, -8.704738220997905",400,1140.09,20%-39%,150,Cabine alta


### 2.5 Create FACT_CONSUMPTION

In [104]:
# Process consumption data
fact_consumption = df_consumption.copy()

# Rename datetime column - handle mixed timezones
if 'Date/Time' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Date/Time'], utc=True)
    fact_consumption['datetime'] = fact_consumption['datetime'].dt.tz_localize(None)
elif 'Period' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Period'], utc=True)
elif 'Date' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Date'])

# Extract hour information from the datetime
if 'datetime' in fact_consumption.columns and pd.api.types.is_datetime64_any_dtype(fact_consumption['datetime']):
    fact_consumption['date'] = fact_consumption['datetime'].dt.date
    fact_consumption['hour'] = fact_consumption['datetime'].dt.hour

# Map municipality to location_id
fact_consumption = fact_consumption.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='Municipality',
    right_on='municipality_name',
    how='left'
)

# Map datetime to time_id (hour level)
if 'datetime' in fact_consumption.columns and pd.api.types.is_datetime64_any_dtype(fact_consumption['datetime']):
    fact_consumption = fact_consumption.merge(
        dim_time[['datetime', 'hour_id', 'day_id']],
        on='datetime',
        how='left'
    )
    
    # Map to weather (using day_id)
    fact_consumption = fact_consumption.merge(
        dim_weather[['time_id', 'weather_id']],
        left_on='day_id',
        right_on='time_id',
        how='left'
    )

# Rename energy consumed column
if 'Energy Consumed' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Energy Consumed'], errors='coerce')
elif 'Value' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Value'], errors='coerce')
elif 'value' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['value'], errors='coerce')

# Select fact table columns  
available_cols = ['location_id', 'hour_id', 'day_id', 'weather_id', 'energy_consumed']
selected_cols = [col for col in available_cols if col in fact_consumption.columns]
fact_consumption = fact_consumption[selected_cols]

# Remove rows with missing key references
fact_consumption = fact_consumption.dropna(subset=['location_id', 'energy_consumed'])

logger.info(f"✓ FACT_CONSUMPTION created: {len(fact_consumption)} rows")
display(fact_consumption.head(10))

2026-04-03 23:49:45,088 - INFO - ✓ FACT_CONSUMPTION created: 1027232 rows


,location_id,hour_id,day_id,weather_id,energy_consumed
0,11,41620,20362,1370,0.0
1,11,41711,20365,1373,0.0
2,11,42239,20387,1395,0.0
3,11,42123,20383,1391,0.0
4,11,41843,20371,1379,0.0
5,11,41716,20366,1374,0.0
6,11,41736,20367,1375,0.0
7,11,41906,20374,1382,0.0
8,11,42178,20385,1393,0.0
9,11,42038,20379,1387,0.0


### 2.6 Create FACT_CHARGING

In [105]:
# Process charging data
fact_charging = df_charging.copy()

# Rename charging_points to energy_consumed for consistency
fact_charging['energy_consumed'] = fact_charging['charging_points']

# Map location to location_id
fact_charging = fact_charging.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='location',
    right_on='municipality_name',
    how='left'
)

# Create datetime from year and month
fact_charging['datetime'] = pd.to_datetime(
    fact_charging['year'].astype(str) + '-' +
    fact_charging['month'].astype(str) + '-01'
)

# Map to month level time_id
fact_charging = fact_charging.merge(
    dim_time[['datetime', 'month_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# Select fact table columns
fact_charging = fact_charging[[
    'location_id',
    'month_id',
    'outlet_type',
    'energy_consumed'
]]

# Remove rows with missing key references
fact_charging = fact_charging.dropna(subset=['location_id', 'month_id', 'energy_consumed'])

logger.info(f"✓ FACT_CHARGING created: {len(fact_charging)} rows")
display(fact_charging.head(10))

2026-04-03 23:49:45,127 - INFO - ✓ FACT_CHARGING created: 3232 rows


,location_id,month_id,outlet_type,energy_consumed
0,2.0,1,Total,4.0
1,2.0,1,Mennekes,4.0
6,3.0,1,Total,2.0
7,3.0,1,Mennekes,2.0
12,4.0,1,Total,3.0
13,4.0,1,Mennekes,1.0
14,4.0,1,CHAdeMO,1.0
15,4.0,1,CCS,1.0
18,5.0,1,Total,14.0
19,5.0,1,Mennekes,10.0


### 2.7 Create FACT_POPULATION

In [106]:
# Process population data
fact_population = df_pop.copy()

# Remove rows with missing year or region
fact_population = fact_population.dropna(subset=['year', 'region_name'])

# Map region to location_id
fact_population = fact_population.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='region_name',
    right_on='municipality_name',
    how='left'
)

# Create datetime from year (01/01)
fact_population['datetime'] = pd.to_datetime(
    fact_population['year'].astype(str) + '-01-01'
)

# Map to year level time_id
fact_population = fact_population.merge(
    dim_time[['datetime', 'time_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# Convert measures to numeric
fact_population['density'] = pd.to_numeric(fact_population['density'], errors='coerce')
fact_population['growth_rate'] = pd.to_numeric(fact_population['growth_rate'], errors='coerce')

# Select fact table columns
fact_population = fact_population[[
    'location_id',
    'time_id',
    'density',
    'growth_rate'
]]

# Remove rows with missing key references
fact_population = fact_population.dropna(subset=['location_id', 'time_id'])

logger.info(f"✓ FACT_POPULATION created: {len(fact_population)} rows")
display(fact_population.head(10))

2026-04-03 23:49:45,175 - INFO - ✓ FACT_POPULATION created: 72 rows


,location_id,time_id,density,growth_rate
0,1,4,NaN,NaN
1,2,4,NaN,NaN
2,3,4,NaN,NaN
3,4,4,1284.0,NaN
4,5,4,1744.0,NaN
5,6,4,NaN,NaN
6,7,4,NaN,NaN
7,8,4,NaN,NaN
8,9,4,NaN,NaN
9,10,4,NaN,NaN


## Phase 3: Validation

Data quality checks and referential integrity

In [107]:
logger.info("=" * 60)
logger.info("PHASE 3: VALIDATION")
logger.info("=" * 60)

validation_report = []

def check_nulls(df, table_name, critical_columns):
    """Check for null values in critical columns"""
    null_count = df[critical_columns].isnull().sum()
    if null_count.sum() > 0:
        logger.warning(f"{table_name} - NULL values found:")
        for col, count in null_count[null_count > 0].items():
            logger.warning(f"  {col}: {count} nulls")
        validation_report.append(f"⚠ {table_name}: {null_count.sum()} null values")
    else:
        logger.info(f"✓ {table_name}: No null values in critical columns")
        validation_report.append(f"✓ {table_name}: No null values")

# Dimension validation
logger.info("\n--- DIMENSION VALIDATION ---")
check_nulls(dim_location, "DIM_LOCATION", ['location_id', 'municipality_code'])
check_nulls(dim_time, "DIM_TIME", ['time_id', 'year'])
check_nulls(dim_weather, "DIM_WEATHER", ['weather_id'])
check_nulls(dim_substations, "DIM_SUBSTATIONS", ['substation_id'])

# Fact validation
logger.info("\n--- FACT TABLE VALIDATION ---")
check_nulls(fact_consumption, "FACT_CONSUMPTION", ['location_id', 'hour_id', 'energy_consumed'])
check_nulls(fact_charging, "FACT_CHARGING", ['location_id', 'month_id', 'energy_consumed'])
check_nulls(fact_population, "FACT_POPULATION", ['location_id', 'time_id'])

2026-04-03 23:49:45,223 - INFO - ============================================================
2026-04-03 23:49:45,224 - INFO - PHASE 3: VALIDATION
2026-04-03 23:49:45,224 - INFO - ============================================================
2026-04-03 23:49:45,225 - INFO - 
--- DIMENSION VALIDATION ---
2026-04-03 23:49:45,227 - INFO - ✓ DIM_LOCATION: No null values in critical columns
2026-04-03 23:49:45,229 - INFO - ✓ DIM_TIME: No null values in critical columns
2026-04-03 23:49:45,231 - INFO - ✓ DIM_WEATHER: No null values in critical columns
2026-04-03 23:49:45,234 - INFO - ✓ DIM_SUBSTATIONS: No null values in critical columns
2026-04-03 23:49:45,235 - INFO - 
--- FACT TABLE VALIDATION ---
2026-04-03 23:49:45,252 - INFO - ✓ FACT_CONSUMPTION: No null values in critical columns
2026-04-03 23:49:45,254 - INFO - ✓ FACT_CHARGING: No null values in critical columns
2026-04-03 23:49:45,256 - INFO - ✓ FACT_POPULATION: No null values in critical columns


In [108]:
# Referential Integrity Checks
logger.info("\n--- REFERENTIAL INTEGRITY ---")

def check_foreign_keys(fact_df, dim_df, fact_col, dim_col, table_names):
    """Check if all FK values exist in dimension"""
    missing = fact_df[~fact_df[fact_col].isin(dim_df[dim_col])]
    if len(missing) > 0:
        logger.warning(f"{table_names[0]} → {table_names[1]}: {len(missing)} orphan records")
        validation_report.append(f"⚠ {table_names[0]} → {table_names[1]}: {len(missing)} orphan FK")
    else:
        logger.info(f"✓ {table_names[0]} → {table_names[1]}: FK integrity OK")
        validation_report.append(f"✓ {table_names[0]} → {table_names[1]}: FK OK ({len(fact_df)} rows)")

# Check consumption FKs
check_foreign_keys(fact_consumption, dim_location, 'location_id', 'location_id', 
                   ['FACT_CONSUMPTION', 'DIM_LOCATION'])
check_foreign_keys(fact_consumption, dim_time, 'hour_id', 'hour_id', 
                   ['FACT_CONSUMPTION', 'DIM_TIME'])
check_foreign_keys(fact_consumption, dim_weather, 'weather_id', 'weather_id', 
                   ['FACT_CONSUMPTION', 'DIM_WEATHER'])

# Check charging FKs
check_foreign_keys(fact_charging, dim_location, 'location_id', 'location_id', 
                   ['FACT_CHARGING', 'DIM_LOCATION'])
check_foreign_keys(fact_charging, dim_time, 'month_id', 'month_id', 
                   ['FACT_CHARGING', 'DIM_TIME'])

# Check population FKs
check_foreign_keys(fact_population, dim_location, 'location_id', 'location_id', 
                   ['FACT_POPULATION', 'DIM_LOCATION'])
check_foreign_keys(fact_population, dim_time, 'time_id', 'time_id', 
                   ['FACT_POPULATION', 'DIM_TIME'])

2026-04-03 23:49:45,272 - INFO - 
--- REFERENTIAL INTEGRITY ---
2026-04-03 23:49:45,290 - INFO - ✓ FACT_CONSUMPTION → DIM_LOCATION: FK integrity OK
2026-04-03 23:49:45,302 - INFO - ✓ FACT_CONSUMPTION → DIM_TIME: FK integrity OK
2026-04-03 23:49:45,312 - INFO - ✓ FACT_CONSUMPTION → DIM_WEATHER: FK integrity OK
2026-04-03 23:49:45,315 - INFO - ✓ FACT_CHARGING → DIM_LOCATION: FK integrity OK
2026-04-03 23:49:45,318 - INFO - ✓ FACT_CHARGING → DIM_TIME: FK integrity OK


2026-04-03 23:49:45,322 - INFO - ✓ FACT_POPULATION → DIM_LOCATION: FK integrity OK
2026-04-03 23:49:45,326 - INFO - ✓ FACT_POPULATION → DIM_TIME: FK integrity OK


In [109]:
# Summary Statistics
logger.info("\n--- SUMMARY STATISTICS ---")

summary = f"""
DIM_LOCATION:        {len(dim_location):>7} rows | PKs: {len(dim_location['location_id'].unique())} unique
DIM_TIME:            {len(dim_time):>7} rows | PKs: {len(dim_time['time_id'].unique())} unique
DIM_WEATHER:         {len(dim_weather):>7} rows | PKs: {len(dim_weather['weather_id'].unique())} unique
DIM_SUBSTATIONS:     {len(dim_substations):>7} rows | PKs: {len(dim_substations['substation_id'].unique())} unique

FACT_CONSUMPTION:    {len(fact_consumption):>7} rows | Energy: {fact_consumption['energy_consumed'].sum():.2f} kWh
FACT_CHARGING:       {len(fact_charging):>7} rows | Energy: {fact_charging['energy_consumed'].sum():.2f} kWh
FACT_POPULATION:     {len(fact_population):>7} rows | Locations: {len(fact_population['location_id'].unique())}
"""

logger.info(summary)
validation_report.append(summary)

2026-04-03 23:49:45,352 - INFO - 
--- SUMMARY STATISTICS ---
2026-04-03 23:49:45,358 - INFO - 
DIM_LOCATION:             18 rows | PKs: 18 unique
DIM_TIME:              45217 rows | PKs: 6 unique
DIM_WEATHER:            1520 rows | PKs: 1520 unique
DIM_SUBSTATIONS:        8412 rows | PKs: 8412 unique

FACT_CONSUMPTION:    1027232 rows | Energy: 29865932.19 kWh
FACT_CHARGING:          3232 rows | Energy: 231884.00 kWh
FACT_POPULATION:          72 rows | Locations: 18



## Phase 4: Loading

Export to CSV files

In [110]:
logger.info("=" * 60)
logger.info("PHASE 4: LOADING")
logger.info("=" * 60)

# Export Dimensions
logger.info("\nExporting dimension tables...")
dim_location.to_csv(os.path.join(DIMS_DIR, 'dim_location.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_location.csv")

dim_time.to_csv(os.path.join(DIMS_DIR, 'dim_time.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_time.csv")

dim_weather.to_csv(os.path.join(DIMS_DIR, 'dim_weather.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_weather.csv")

dim_substations.to_csv(os.path.join(DIMS_DIR, 'dim_substations.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_substations.csv")

# Export Facts
logger.info("\nExporting fact tables...")
fact_consumption.to_csv(os.path.join(FACTS_DIR, 'fact_consumption.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_consumption.csv")

fact_charging.to_csv(os.path.join(FACTS_DIR, 'fact_charging.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_charging.csv")

fact_population.to_csv(os.path.join(FACTS_DIR, 'fact_population.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_population.csv")

logger.info("\n✓ All tables exported successfully")

2026-04-03 23:49:45,383 - INFO - ============================================================
2026-04-03 23:49:45,384 - INFO - PHASE 4: LOADING
2026-04-03 23:49:45,385 - INFO - ============================================================
2026-04-03 23:49:45,386 - INFO - 
Exporting dimension tables...
2026-04-03 23:49:45,389 - INFO - ✓ ./dw_output\dimensions/dim_location.csv
2026-04-03 23:49:45,530 - INFO - ✓ ./dw_output\dimensions/dim_time.csv
2026-04-03 23:49:45,537 - INFO - ✓ ./dw_output\dimensions/dim_weather.csv
2026-04-03 23:49:45,557 - INFO - ✓ ./dw_output\dimensions/dim_substations.csv
2026-04-03 23:49:45,558 - INFO - 
Exporting fact tables...
2026-04-03 23:49:46,621 - INFO - ✓ ./dw_output\facts/fact_consumption.csv
2026-04-03 23:49:46,627 - INFO - ✓ ./dw_output\facts/fact_charging.csv
2026-04-03 23:49:46,630 - INFO - ✓ ./dw_output\facts/fact_population.csv
2026-04-03 23:49:46,630 - INFO - 
✓ All tables exported successfully


In [111]:
# Generate Validation Report
report_path = os.path.join(LOGS_DIR, 'etl_validation_report.txt')

with open(report_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("DATA WAREHOUSE ETL VALIDATION REPORT\n")
    f.write("="*60 + "\n\n")
    
    f.write(f"Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("VALIDATION RESULTS:\n")
    f.write("-" * 60 + "\n")
    for item in validation_report:
        f.write(str(item) + "\n")
    
    f.write("\n" + "-" * 60 + "\n")
    f.write(f"ETL EXECUTION: SUCCESS\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n")

logger.info(f"✓ Validation report saved: {report_path}")
logger.info("="*60)
logger.info("ETL PROCESS COMPLETED SUCCESSFULLY")
logger.info("="*60)

2026-04-03 23:49:46,642 - INFO - ✓ Validation report saved: ./dw_output\logs\etl_validation_report.txt
2026-04-03 23:49:46,644 - INFO - ============================================================
2026-04-03 23:49:46,645 - INFO - ETL PROCESS COMPLETED SUCCESSFULLY
2026-04-03 23:49:46,646 - INFO - ============================================================


## Summary

### Dimensional Bus Matrix Implementation

| Data Mart | Star | Dimensions Used |
|-----------|------|------------------|
| Energy | Consumption | Location, Time, Weather, Substations |
| Mobility | Charging | Location, Time |
| Demographics | Population | Location, Time |

### Output Files

**Dimensions:**
- `dimensions/dim_location.csv` - Geographic hierarchy
- `dimensions/dim_time.csv` - Multi-level temporal dimension
- `dimensions/dim_weather.csv` - Daily weather conditions
- `dimensions/dim_substations.csv` - Electrical infrastructure

**Facts:**
- `facts/fact_consumption.csv` - Hourly energy consumption
- `facts/fact_charging.csv` - Monthly EV charging activity
- `facts/fact_population.csv` - Yearly demographic indicators

### Next Steps

1. **Load to Database**: Create SQL schema and load CSVs to database
2. **Add Indexes**: Create indexes on PK, FK, and commonly filtered columns
3. **OLAP Cubes**: Build aggregated views for reporting
4. **Test Queries**: Run dimensional analysis queries
5. **Documentation**: Create data dictionary and business glossary